# 05 SHAP and Interpretability Analysis

This notebook applies interpretability methods to the tennis match prediction models developed earlier in the project.

The purpose of this notebook is to move beyond predictive accuracy alone and examine how different models generate their predictions, with particular attention to the role of serve and return statistics across different match contexts.

In line with the methodology, the notebook focuses on:

- global explanation patterns across models
- surface-specific explanation patterns
- local match-level case studies
- comparison of explanations in contexts where rankings alone may be insufficient

## 1. Research Focus of This Notebook

The primary aim of this notebook is to investigate how interpretable machine learning techniques can be used to explain tennis match outcome prediction models.

Although global model behaviour is expected to be influenced strongly by broad player-strength indicators such as Elo and ranking-related variables, the analysis in this notebook places particular emphasis on whether serve and return statistics contribute to predictions in different surfaces and match contexts.

This directly supports the dissertation's interpretability objectives by examining:

- which features influence predictions overall
- whether explanation patterns differ across model families
- whether surface-specific contexts produce different explanatory patterns
- whether local explanations reveal meaningful match-specific factors beyond rankings alone

## 2. Load Data and Recreate the Modelling Inputs

The final match-level feature dataset is loaded here. The same curated feature set, temporal split, and preprocessing steps used during modelling are recreated to ensure consistency between prediction and explanation.

In [34]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [35]:
PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR = PROJECT_DIR / "data"
FEATURE_DIR = DATA_DIR / "features"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Feature directory:", FEATURE_DIR)
print("Output directory:", OUTPUT_DIR)

Project directory: /Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project
Feature directory: /Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/data/features
Output directory: /Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/outputs


In [36]:
df_model = pd.read_csv(
    FEATURE_DIR / "match_level_features.csv",
    parse_dates=["tourney_date"]
)

print("Dataset shape:", df_model.shape)
df_model.head()

Dataset shape: (29394, 199)


,tourney_date,tourney_id,match_num,A_tourney_name,A_surface,A_draw_size,A_tourney_level,A_score,A_best_of,A_round,...,B_ELOSTD52W,B_player_order,target,RankDifference,ELODifference,SurfaceELODifference,RankChange26W_Diff,RankSTD52W_Diff,ELOChange26W_Diff,ELOSTD52W_Diff
0,2013-01-07,2013-301,1,Auckland,Hard,28,A,6-3 2-6 6-2,3,R32,...,NaN,1,1,17.0,0.0,0.0,-2.0,-16.059859,NaN,NaN
1,2013-01-07,2013-301,2,Auckland,Hard,28,A,6-3 6-3,3,R32,...,NaN,1,0,9.0,0.0,0.0,-26.0,-0.874811,NaN,NaN
2,2013-01-07,2013-301,3,Auckland,Hard,28,A,7-6(5) 6-2,3,R32,...,NaN,1,1,-18.0,0.0,0.0,-23.0,-30.345402,NaN,NaN
3,2013-01-07,2013-301,4,Auckland,Hard,28,A,3-6 6-3 7-5,3,R32,...,NaN,1,0,-16.0,0.0,0.0,52.0,-29.032247,NaN,NaN
4,2013-01-07,2013-301,5,Auckland,Hard,28,A,6-7(2) 6-3 6-4,3,R32,...,NaN,1,0,-34.0,0.0,0.0,-129.0,52.664222,NaN,NaN


## 3. Recreate the Curated Feature Set

The same pre-match feature set used during model training is reconstructed here so that all explanations correspond directly to the features seen by the predictive models.

In [37]:
model_features = [
    # Context
    "A_draw_size", "A_best_of",
    "A_player_age", "B_player_age",
    "A_player_ht", "B_player_ht",

    # Ranking features
    "A_CurrentRanking", "B_CurrentRanking",
    "A_RankingCareerHigh", "B_RankingCareerHigh",
    "A_WeekSinceCareerHigh", "B_WeekSinceCareerHigh",
    "A_RankChange26W", "B_RankChange26W",
    "A_RankSTD52W", "B_RankSTD52W",
    "RankDifference",
    "RankChange26W_Diff",
    "RankSTD52W_Diff",

    # Elo features
    "A_ELO", "B_ELO",
    "A_ELOHigh", "B_ELOHigh",
    "A_SurfaceELO", "B_SurfaceELO",
    "A_ELOChange26W", "B_ELOChange26W",
    "A_ELOSTD52W", "B_ELOSTD52W",
    "ELODifference",
    "SurfaceELODifference",
    "ELOChange26W_Diff",
    "ELOSTD52W_Diff",

    # L15 rolling performance
    "A_AceRateL15", "B_AceRateL15",
    "A_DFRateL15", "B_DFRateL15",
    "A_FirstServeInPctL15", "B_FirstServeInPctL15",
    "A_FirstServeWinPctL15", "B_FirstServeWinPctL15",
    "A_SecondServeWinPctL15", "B_SecondServeWinPctL15",
    "A_ServicePointsWonPctL15", "B_ServicePointsWonPctL15",
    "A_BreakPointsSavedPctL15", "B_BreakPointsSavedPctL15",
    "A_BreakPointsFacedRateL15", "B_BreakPointsFacedRateL15",
    "A_ReturnPointsWonPctL15", "B_ReturnPointsWonPctL15",
    "A_BreakPointsConvertedPctL15", "B_BreakPointsConvertedPctL15",
    "A_BreakPointsEarnedRateL15", "B_BreakPointsEarnedRateL15",

    # 52W rolling performance
    "A_FirstServeInPct52W", "B_FirstServeInPct52W",
    "A_FirstServeWinPct52W", "B_FirstServeWinPct52W",
    "A_SecondServeWinPct52W", "B_SecondServeWinPct52W",
    "A_ServicePointsWonPct52W", "B_ServicePointsWonPct52W",
    "A_ReturnPointsWonPct52W", "B_ReturnPointsWonPct52W",
    "A_BreakPointsConvertedPct52W", "B_BreakPointsConvertedPct52W",
    "A_BreakPointsEarnedRate52W", "B_BreakPointsEarnedRate52W",
]

print("Number of features:", len(model_features))

Number of features: 69


## 4. Recreate the Time-Based Train/Test Split

To preserve temporal validity, the dataset is split chronologically into a training period and a held-out test period. This mirrors the modelling setup used previously and ensures that explanations are generated for out-of-sample predictions.

In [38]:
X = df_model[model_features]
y = df_model["target"]

cutoff_date = "2021-01-01"

train_mask = df_model["tourney_date"] < cutoff_date
test_mask = df_model["tourney_date"] >= cutoff_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

df_train = df_model.loc[X_train.index].copy()
df_test = df_model.loc[X_test.index].copy()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("df_test shape:", df_test.shape)

Train shape: (19004, 69)
Test shape: (10390, 69)
df_test shape: (10390, 199)


## 5. Impute Missing Values

Missing values are handled using median imputation fitted on the training data only. This reproduces the preprocessing procedure used during model training.

In [39]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

X_train_imputed_df = pd.DataFrame(
    X_train_imputed,
    columns=model_features,
    index=X_train.index
)

X_test_imputed_df = pd.DataFrame(
    X_test_imputed,
    columns=model_features,
    index=X_test.index
)

print("Imputation complete")
print(X_train_imputed_df.shape)
print(X_test_imputed_df.shape)

Imputation complete
(19004, 69)
(10390, 69)


## 6. Train Explanation Models

To align with the comparative methodology of the dissertation, multiple model families are trained for interpretability analysis. These represent increasing model complexity:

- Logistic Regression
- Random Forest
- XGBoost

This allows comparison of explanation patterns across simpler and more complex predictive models.

In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [41]:
from sklearn.preprocessing import StandardScaler

# --- Logistic Regression (with scaling) ---
lr_scaler = StandardScaler()

X_train_scaled = lr_scaler.fit_transform(X_train_imputed)
X_test_scaled = lr_scaler.transform(X_test_imputed)

lr_clf = LogisticRegression(
    max_iter=2000,
    random_state=42
)

lr_clf.fit(X_train_scaled, y_train)

# --- Random Forest ---
rf_clf = RandomForestClassifier(
    n_estimators=600,
    max_depth=25,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(X_train_imputed, y_train)

# --- XGBoost ---
xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)
xgb_clf.fit(X_train_imputed, y_train)

print("All explanation models trained successfully")

All explanation models trained successfully


## 7. Compute Global Explanations Across Models

Global explanations are used first to examine which features influence predictions on average across the test set.

This section compares explanation patterns across:

- Logistic Regression
- Random Forest
- XGBoost

Because broad player-strength variables such as Elo and ranking-related features are expected to dominate global explanations, this section establishes the general pattern before later sections focus more specifically on serve and return behaviour across different surfaces and case studies.

In [42]:
import shap

# --- Logistic Regression global importance from coefficients ---
lr_global_importance = pd.DataFrame({
    "Feature": model_features,
    "Importance": np.abs(lr_clf.coef_[0])
}).sort_values(by="Importance", ascending=False)

# --- Random Forest global importance from feature importances ---
rf_global_importance = pd.DataFrame({
    "Feature": model_features,
    "Importance": rf_clf.feature_importances_
}).sort_values(by="Importance", ascending=False)

# --- XGBoost global importance from mean absolute SHAP values ---
xgb_explainer = shap.TreeExplainer(xgb_clf)
xgb_shap_values = xgb_explainer.shap_values(X_test_imputed_df)

xgb_global_importance = pd.DataFrame({
    "Feature": model_features,
    "Importance": np.abs(xgb_shap_values).mean(axis=0)
}).sort_values(by="Importance", ascending=False)

print("Global explanation objects prepared")

Global explanation objects prepared


In [43]:
print("Top 15 Logistic Regression Features")
display(lr_global_importance.head(15))

print("Top 15 Random Forest Features")
display(rf_global_importance.head(15))

print("Top 15 XGBoost Features")
display(xgb_global_importance.head(15))

Top 15 Logistic Regression Features


,Feature,Importance
62,B_ServicePointsWonPct52W,0.522665
61,A_ServicePointsWonPct52W,0.371648
58,B_FirstServeWinPct52W,0.347992
9,B_RankingCareerHigh,0.336223
44,B_ServicePointsWonPctL15,0.329871
40,B_FirstServeWinPctL15,0.281018
8,A_RankingCareerHigh,0.249062
57,A_FirstServeWinPct52W,0.229788
30,SurfaceELODifference,0.193742
60,B_SecondServeWinPct52W,0.172337


Top 15 Random Forest Features


,Feature,Importance
29,ELODifference,0.084956
16,RankDifference,0.081698
30,SurfaceELODifference,0.067146
6,A_CurrentRanking,0.030481
9,B_RankingCareerHigh,0.027446
7,B_CurrentRanking,0.027433
19,A_ELO,0.027146
20,B_ELO,0.027103
8,A_RankingCareerHigh,0.023703
18,RankSTD52W_Diff,0.023456


Top 15 XGBoost Features


,Feature,Importance
16,RankDifference,0.213029
29,ELODifference,0.211016
30,SurfaceELODifference,0.191972
8,A_RankingCareerHigh,0.121278
2,A_player_age,0.099576
3,B_player_age,0.096381
9,B_RankingCareerHigh,0.063367
11,B_WeekSinceCareerHigh,0.061220
6,A_CurrentRanking,0.051825
64,B_ReturnPointsWonPct52W,0.045354


## 8. Filter Out Broad Strength Variables for Tennis-Specific Interpretation

Global explanation results are expected to be dominated by broad player-strength variables such as Elo and ranking-related metrics. To better examine tennis-specific performance signals, a filtered view is created in which these broad strength indicators are removed from the ranked feature lists.

This does not change the underlying models. It only provides a second interpretive lens focused more explicitly on serve, return, and performance-trend statistics.

In [44]:
strength_features = [
    # ranking level features
    "RankDifference",
    "A_CurrentRanking", "B_CurrentRanking",
    "A_RankingCareerHigh", "B_RankingCareerHigh",
    "A_WeekSinceCareerHigh", "B_WeekSinceCareerHigh",

    # ranking trend / volatility
    "A_RankChange26W", "B_RankChange26W",
    "RankChange26W_Diff",
    "A_RankSTD52W", "B_RankSTD52W",
    "RankSTD52W_Diff",

    # Elo level features
    "A_ELO", "B_ELO",
    "A_ELOHigh", "B_ELOHigh",
    "A_SurfaceELO", "B_SurfaceELO",
    "ELODifference",
    "SurfaceELODifference",

    # Elo trend / volatility
    "A_ELOChange26W", "B_ELOChange26W",
    "ELOChange26W_Diff",
    "A_ELOSTD52W", "B_ELOSTD52W",
    "ELOSTD52W_Diff",
]

In [45]:
lr_global_filtered = lr_global_importance[
    ~lr_global_importance["Feature"].isin(strength_features)
].reset_index(drop=True)

rf_global_filtered = rf_global_importance[
    ~rf_global_importance["Feature"].isin(strength_features)
].reset_index(drop=True)

xgb_global_filtered = xgb_global_importance[
    ~xgb_global_importance["Feature"].isin(strength_features)
].reset_index(drop=True)

print("Top 15 Logistic Regression Features (excluding strength features)")
display(lr_global_filtered.head(15))

print("Top 15 Random Forest Features (excluding strength features)")
display(rf_global_filtered.head(15))

print("Top 15 XGBoost Features (excluding strength features)")
display(xgb_global_filtered.head(15))

Top 15 Logistic Regression Features (excluding strength features)


,Feature,Importance
0,B_ServicePointsWonPct52W,0.522665
1,A_ServicePointsWonPct52W,0.371648
2,B_FirstServeWinPct52W,0.347992
3,B_ServicePointsWonPctL15,0.329871
4,B_FirstServeWinPctL15,0.281018
5,A_FirstServeWinPct52W,0.229788
6,B_SecondServeWinPct52W,0.172337
7,B_player_age,0.169049
8,A_ServicePointsWonPctL15,0.166805
9,A_ReturnPointsWonPctL15,0.163788


Top 15 Random Forest Features (excluding strength features)


,Feature,Importance
0,A_ServicePointsWonPct52W,0.013685
1,B_ServicePointsWonPct52W,0.012579
2,A_SecondServeWinPct52W,0.012396
3,B_ServicePointsWonPctL15,0.012192
4,A_ServicePointsWonPctL15,0.011822
5,A_ReturnPointsWonPct52W,0.011088
6,A_BreakPointsFacedRateL15,0.011005
7,B_ReturnPointsWonPct52W,0.010766
8,A_ReturnPointsWonPctL15,0.010583
9,A_SecondServeWinPctL15,0.010524


Top 15 XGBoost Features (excluding strength features)


,Feature,Importance
0,A_player_age,0.099576
1,B_player_age,0.096381
2,B_ReturnPointsWonPct52W,0.045354
3,A_ReturnPointsWonPctL15,0.042738
4,A_ReturnPointsWonPct52W,0.041797
5,B_BreakPointsEarnedRate52W,0.039736
6,B_ServicePointsWonPct52W,0.039316
7,B_BreakPointsFacedRateL15,0.038801
8,B_AceRateL15,0.036476
9,A_SecondServeWinPctL15,0.033136


In [46]:
lr_global_filtered.to_csv(OUTPUT_DIR / "lr_global_filtered.csv", index=False)
rf_global_filtered.to_csv(OUTPUT_DIR / "rf_global_filtered.csv", index=False)
xgb_global_filtered.to_csv(OUTPUT_DIR / "xgb_global_filtered.csv", index=False)

print("Saved global filtered feature tables")

Saved global filtered feature tables


## 9. Surface-Specific SHAP Analysis

The cross-model comparison above shows that tennis-specific features remain important even after broad strength variables are removed. The next step is to investigate whether these explanatory patterns vary across playing surfaces.

Because SHAP is used as the primary explanation framework for the most complex model, this section focuses on XGBoost and compares grass-court and clay-court matches separately.

The main objective is to examine whether serve-related features become more prominent on grass, and whether return-related features become more prominent on clay.

In [47]:
# Surface masks from the held-out test set
grass_mask = df_test["A_surface"] == "Grass"
clay_mask = df_test["A_surface"] == "Clay"

print("Grass matches:", grass_mask.sum())
print("Clay matches:", clay_mask.sum())

Grass matches: 1230
Clay matches: 3188


In [48]:
# Convert XGBoost SHAP values into a DataFrame for easier filtering
xgb_shap_df = pd.DataFrame(
    xgb_shap_values,
    columns=model_features,
    index=X_test_imputed_df.index
)

# Subset SHAP values by surface
xgb_shap_grass = xgb_shap_df.loc[grass_mask]
xgb_shap_clay = xgb_shap_df.loc[clay_mask]

print("Grass SHAP shape:", xgb_shap_grass.shape)
print("Clay SHAP shape:", xgb_shap_clay.shape)

Grass SHAP shape: (1230, 69)
Clay SHAP shape: (3188, 69)


In [49]:
# Mean absolute SHAP importance by surface
grass_importance = pd.DataFrame({
    "Feature": model_features,
    "Importance": np.abs(xgb_shap_grass).mean(axis=0)
}).sort_values(by="Importance", ascending=False)

clay_importance = pd.DataFrame({
    "Feature": model_features,
    "Importance": np.abs(xgb_shap_clay).mean(axis=0)
}).sort_values(by="Importance", ascending=False)

print("Top 15 Grass Features (all)")
display(grass_importance.head(15))

print("Top 15 Clay Features (all)")
display(clay_importance.head(15))

Top 15 Grass Features (all)


,Feature,Importance
ELODifference,ELODifference,0.222697
RankDifference,RankDifference,0.221702
SurfaceELODifference,SurfaceELODifference,0.152887
A_RankingCareerHigh,A_RankingCareerHigh,0.121840
A_player_age,A_player_age,0.105746
B_player_age,B_player_age,0.087516
B_RankingCareerHigh,B_RankingCareerHigh,0.064996
B_WeekSinceCareerHigh,B_WeekSinceCareerHigh,0.061834
A_CurrentRanking,A_CurrentRanking,0.046559
B_ReturnPointsWonPct52W,B_ReturnPointsWonPct52W,0.043862


Top 15 Clay Features (all)


,Feature,Importance
RankDifference,RankDifference,0.216857
ELODifference,ELODifference,0.207307
SurfaceELODifference,SurfaceELODifference,0.186803
A_RankingCareerHigh,A_RankingCareerHigh,0.125798
A_player_age,A_player_age,0.098520
B_player_age,B_player_age,0.098436
B_RankingCareerHigh,B_RankingCareerHigh,0.064974
B_WeekSinceCareerHigh,B_WeekSinceCareerHigh,0.061634
A_CurrentRanking,A_CurrentRanking,0.051880
A_WeekSinceCareerHigh,A_WeekSinceCareerHigh,0.044851


## 10. Filtered Surface-Specific Tennis Features

As in the previous section, broad strength indicators are removed from the ranked lists so that surface-specific tennis features can be compared more directly.

In [50]:
grass_filtered = grass_importance[
    ~grass_importance["Feature"].isin(strength_features)
].reset_index(drop=True)

clay_filtered = clay_importance[
    ~clay_importance["Feature"].isin(strength_features)
].reset_index(drop=True)

print("Top 15 Grass Features (excluding strength features)")
display(grass_filtered.head(15))

print("Top 15 Clay Features (excluding strength features)")
display(clay_filtered.head(15))

Top 15 Grass Features (excluding strength features)


,Feature,Importance
0,A_player_age,0.105746
1,B_player_age,0.087516
2,B_ReturnPointsWonPct52W,0.043862
3,A_ReturnPointsWonPctL15,0.042730
4,B_BreakPointsEarnedRate52W,0.042532
5,A_ReturnPointsWonPct52W,0.042080
6,B_ServicePointsWonPct52W,0.040898
7,B_BreakPointsFacedRateL15,0.038105
8,B_AceRateL15,0.036186
9,A_DFRateL15,0.033184


Top 15 Clay Features (excluding strength features)


,Feature,Importance
0,A_player_age,0.098520
1,B_player_age,0.098436
2,B_ReturnPointsWonPct52W,0.044849
3,A_ReturnPointsWonPctL15,0.044005
4,B_BreakPointsEarnedRate52W,0.043052
5,A_ReturnPointsWonPct52W,0.042989
6,B_AceRateL15,0.041911
7,B_ReturnPointsWonPctL15,0.037082
8,B_BreakPointsFacedRateL15,0.035326
9,B_ServicePointsWonPct52W,0.034567


In [51]:
grass_filtered.to_csv(OUTPUT_DIR / "grass_filtered_shap.csv", index=False)
clay_filtered.to_csv(OUTPUT_DIR / "clay_filtered_shap.csv", index=False)

print("Saved:")
print((OUTPUT_DIR / "grass_filtered_shap.csv").resolve())
print((OUTPUT_DIR / "clay_filtered_shap.csv").resolve())

Saved:
/Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/outputs/grass_filtered_shap.csv
/Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/outputs/clay_filtered_shap.csv


## 11. Preview Candidate Case Study Matches

Before finalising the case studies, the selected matches are inspected using both pre-match feature values and local SHAP outputs. This helps determine which explanatory themes each case best illustrates, such as serve-related influence on grass, return-related influence on clay, or multi-factor explanations in elite matches.

In [52]:
# Predicted probabilities and predicted class from XGBoost
y_pred_proba = xgb_clf.predict_proba(X_test_imputed)[:, 1]
y_pred = xgb_clf.predict(X_test_imputed)

# Clean case-study dataframe aligned to test rows
case_df = df_test.reset_index(drop=True).copy()

case_df["pred_prob_A_win"] = y_pred_proba
case_df["pred_A_win"] = y_pred
case_df["actual_A_win"] = y_test.reset_index(drop=True).values

print("case_df shape:", case_df.shape)
case_df.head()

case_indices = [6465, 8726, 8433]

case_df shape: (10390, 202)


In [53]:
case_preview = case_df.loc[case_indices, [
    "tourney_date",
    "A_surface",
    "A_player_name",
    "B_player_name",
    "A_CurrentRanking",
    "B_CurrentRanking",
    "RankDifference",
    "A_ELO",
    "B_ELO",
    "ELODifference",
    "A_SurfaceELO",
    "B_SurfaceELO",
    "SurfaceELODifference",
    "pred_prob_A_win",
    "actual_A_win"
]].copy()

case_preview

,tourney_date,A_surface,A_player_name,B_player_name,A_CurrentRanking,B_CurrentRanking,RankDifference,A_ELO,B_ELO,ELODifference,A_SurfaceELO,B_SurfaceELO,SurfaceELODifference,pred_prob_A_win,actual_A_win
6465,2023-06-12,Grass,Milos Raonic,Miomir Kecmanovic,738.0,39.0,699.0,1778.467304,1616.606372,161.860932,1736.293574,1530.377685,205.915889,0.552808,1
8726,2024-04-15,Clay,Cameron Norrie,Tomas Martin Etcheverry,31.0,30.0,1.0,1690.427482,1648.864146,41.563336,1694.098713,1636.484242,57.614471,0.600946,0
8433,2024-03-04,Hard,Jannik Sinner,Carlos Alcaraz,3.0,2.0,1.0,2142.003316,2017.449217,124.554100,2121.091761,1947.828950,173.262811,0.602213,0


In [54]:
tennis_feature_preview = case_df.loc[case_indices, [
    "A_player_name",
    "B_player_name",
    "A_surface",

    # Serve-related
    "A_AceRateL15", "B_AceRateL15",
    "A_FirstServeWinPctL15", "B_FirstServeWinPctL15",
    "A_SecondServeWinPctL15", "B_SecondServeWinPctL15",
    "A_ServicePointsWonPctL15", "B_ServicePointsWonPctL15",

    # Return-related
    "A_ReturnPointsWonPctL15", "B_ReturnPointsWonPctL15",
    "A_BreakPointsConvertedPctL15", "B_BreakPointsConvertedPctL15",
    "A_BreakPointsEarnedRateL15", "B_BreakPointsEarnedRateL15",

    # Longer-term supporting stats
    "A_ReturnPointsWonPct52W", "B_ReturnPointsWonPct52W",
    "A_ServicePointsWonPct52W", "B_ServicePointsWonPct52W"
]].copy()

tennis_feature_preview

,A_player_name,B_player_name,A_surface,A_AceRateL15,B_AceRateL15,A_FirstServeWinPctL15,B_FirstServeWinPctL15,A_SecondServeWinPctL15,B_SecondServeWinPctL15,A_ServicePointsWonPctL15,...,A_ReturnPointsWonPctL15,B_ReturnPointsWonPctL15,A_BreakPointsConvertedPctL15,B_BreakPointsConvertedPctL15,A_BreakPointsEarnedRateL15,B_BreakPointsEarnedRateL15,A_ReturnPointsWonPct52W,B_ReturnPointsWonPct52W,A_ServicePointsWonPct52W,B_ServicePointsWonPct52W
6465,Milos Raonic,Miomir Kecmanovic,Grass,0.222320,0.044937,0.817231,0.691821,0.577012,0.558866,0.726363,...,0.363029,0.414431,0.354015,0.430322,0.612720,0.715613,0.306122,0.362081,0.674157,0.631539
8726,Cameron Norrie,Tomas Martin Etcheverry,Clay,0.063874,0.074230,0.709302,0.689133,0.526013,0.484664,0.632777,...,0.414229,0.376783,0.464617,0.429670,0.757901,0.618778,0.380663,0.368136,0.637705,0.637222
8433,Jannik Sinner,Carlos Alcaraz,Hard,0.082355,0.066836,0.791640,0.766940,0.613819,0.570441,0.722459,...,0.424954,0.408017,0.468273,0.391272,0.737490,0.721261,0.409724,0.422318,0.693533,0.680895


In [55]:
X_test_reset = X_test_imputed_df.reset_index(drop=True)

def get_local_shap_table(row_idx, top_n=20):
    row_features = X_test_reset.iloc[row_idx]
    row_shap = xgb_shap_values[row_idx]

    local_df = pd.DataFrame({
        "Feature": X_test_reset.columns,
        "FeatureValue": row_features.values,
        "SHAPValue": row_shap,
        "AbsSHAP": np.abs(row_shap)
    }).sort_values(by="AbsSHAP", ascending=False)

    return local_df.head(top_n)

In [56]:
print("Raonic vs Kecmanovic")
local_raonic = get_local_shap_table(6465, top_n=20)
display(local_raonic)

print("Norrie vs Etcheverry")
local_norrie = get_local_shap_table(8726, top_n=20)
display(local_norrie)

print("Sinner vs Alcaraz")
local_sinner = get_local_shap_table(8433, top_n=20)
display(local_sinner)

Raonic vs Kecmanovic


,Feature,FeatureValue,SHAPValue,AbsSHAP
16,RankDifference,699.000000,-0.582332,0.582332
30,SurfaceELODifference,205.915889,0.431713,0.431713
29,ELODifference,161.860932,0.263157,0.263157
8,A_RankingCareerHigh,3.000000,0.194643,0.194643
65,A_BreakPointsConvertedPct52W,0.600000,-0.176021,0.176021
6,A_CurrentRanking,738.000000,-0.123650,0.123650
2,A_player_age,32.400000,-0.082821,0.082821
11,B_WeekSinceCareerHigh,21.000000,0.079031,0.079031
15,B_RankSTD52W,3.206212,-0.074108,0.074108
55,A_FirstServeInPct52W,0.573034,-0.072728,0.072728


Norrie vs Etcheverry


,Feature,FeatureValue,SHAPValue,AbsSHAP
16,RankDifference,1.000000,-0.118714,0.118714
30,SurfaceELODifference,57.614471,0.115102,0.115102
8,A_RankingCareerHigh,8.000000,0.110429,0.110429
2,A_player_age,28.600000,0.092654,0.092654
11,B_WeekSinceCareerHigh,7.000000,0.079775,0.079775
49,A_ReturnPointsWonPctL15,0.414229,0.074851,0.074851
36,B_DFRateL15,0.012066,0.059576,0.059576
41,A_SecondServeWinPctL15,0.526013,0.049058,0.049058
10,A_WeekSinceCareerHigh,80.000000,0.047714,0.047714
24,B_SurfaceELO,1636.484242,-0.046749,0.046749


Sinner vs Alcaraz


,Feature,FeatureValue,SHAPValue,AbsSHAP
30,SurfaceELODifference,173.262811,0.397208,0.397208
29,ELODifference,124.554100,0.205209,0.205209
16,RankDifference,1.000000,-0.190674,0.190674
8,A_RankingCareerHigh,3.000000,0.187169,0.187169
3,B_player_age,20.800000,-0.184119,0.184119
64,B_ReturnPointsWonPct52W,0.422318,-0.171177,0.171177
9,B_RankingCareerHigh,1.000000,-0.161088,0.161088
2,A_player_age,22.500000,0.139478,0.139478
6,A_CurrentRanking,3.000000,0.138012,0.138012
62,B_ServicePointsWonPct52W,0.680895,-0.119392,0.119392


## 12. Local SHAP Case Studies

Global explanation results provide an overall view of model behaviour, but they do not fully capture the factors influencing individual match predictions. To address this, three local SHAP case studies are examined.

These cases were selected to represent different explanatory contexts:

- a grass-court underdog example
- a clay-court close-ranking example
- an elite closely matched contest

Together, these case studies illustrate how local SHAP explanations can reveal match-specific drivers of prediction beyond broad ranking-based summaries.

In [57]:
# Save the chosen case indices in one place
CASE_RAONIC = 6465
CASE_NORRIE = 8726
CASE_SINNER = 8433

In [58]:
def get_case_metadata(row_idx):
    return case_df.loc[[row_idx], [
        "tourney_date",
        "A_surface",
        "A_player_name",
        "B_player_name",
        "A_CurrentRanking",
        "B_CurrentRanking",
        "RankDifference",
        "A_ELO",
        "B_ELO",
        "ELODifference",
        "A_SurfaceELO",
        "B_SurfaceELO",
        "SurfaceELODifference",
        "pred_prob_A_win",
        "actual_A_win"
    ]].copy()

In [59]:
def get_case_tennis_stats(row_idx):
    return case_df.loc[[row_idx], [
        "A_player_name",
        "B_player_name",
        "A_surface",

        # Serve-related
        "A_AceRateL15", "B_AceRateL15",
        "A_FirstServeWinPctL15", "B_FirstServeWinPctL15",
        "A_SecondServeWinPctL15", "B_SecondServeWinPctL15",
        "A_ServicePointsWonPctL15", "B_ServicePointsWonPctL15",

        # Return-related
        "A_ReturnPointsWonPctL15", "B_ReturnPointsWonPctL15",
        "A_BreakPointsConvertedPctL15", "B_BreakPointsConvertedPctL15",
        "A_BreakPointsEarnedRateL15", "B_BreakPointsEarnedRateL15",

        # Longer-term supporting stats
        "A_ReturnPointsWonPct52W", "B_ReturnPointsWonPct52W",
        "A_ServicePointsWonPct52W", "B_ServicePointsWonPct52W"
    ]].copy()

In [60]:
X_test_reset = X_test_imputed_df.reset_index(drop=True)

def get_local_shap_table(row_idx, top_n=15):
    row_features = X_test_reset.iloc[row_idx]
    row_shap = xgb_shap_values[row_idx]

    local_df = pd.DataFrame({
        "Feature": X_test_reset.columns,
        "FeatureValue": row_features.values,
        "SHAPValue": row_shap,
        "AbsSHAP": np.abs(row_shap)
    }).sort_values(by="AbsSHAP", ascending=False)

    return local_df.head(top_n)

### 12.1 Case Study 1: Grass-Court Underdog Match

This case examines a grass-court match in which the winning player entered with a much weaker official ranking. The purpose of this example is to assess whether local SHAP explanations reveal factors, such as serve-related performance and surface-adjusted strength, that are not captured by ranking alone.

In [61]:
case_raonic_meta = get_case_metadata(CASE_RAONIC)
case_raonic_stats = get_case_tennis_stats(CASE_RAONIC)
case_raonic_shap = get_local_shap_table(CASE_RAONIC, top_n=20)

print("Match metadata")
display(case_raonic_meta)

print("Selected tennis performance features")
display(case_raonic_stats)

print("Top local SHAP contributors")
display(case_raonic_shap)

Match metadata


,tourney_date,A_surface,A_player_name,B_player_name,A_CurrentRanking,B_CurrentRanking,RankDifference,A_ELO,B_ELO,ELODifference,A_SurfaceELO,B_SurfaceELO,SurfaceELODifference,pred_prob_A_win,actual_A_win
6465,2023-06-12,Grass,Milos Raonic,Miomir Kecmanovic,738.0,39.0,699.0,1778.467304,1616.606372,161.860932,1736.293574,1530.377685,205.915889,0.552808,1


Selected tennis performance features


,A_player_name,B_player_name,A_surface,A_AceRateL15,B_AceRateL15,A_FirstServeWinPctL15,B_FirstServeWinPctL15,A_SecondServeWinPctL15,B_SecondServeWinPctL15,A_ServicePointsWonPctL15,...,A_ReturnPointsWonPctL15,B_ReturnPointsWonPctL15,A_BreakPointsConvertedPctL15,B_BreakPointsConvertedPctL15,A_BreakPointsEarnedRateL15,B_BreakPointsEarnedRateL15,A_ReturnPointsWonPct52W,B_ReturnPointsWonPct52W,A_ServicePointsWonPct52W,B_ServicePointsWonPct52W
6465,Milos Raonic,Miomir Kecmanovic,Grass,0.22232,0.044937,0.817231,0.691821,0.577012,0.558866,0.726363,...,0.363029,0.414431,0.354015,0.430322,0.61272,0.715613,0.306122,0.362081,0.674157,0.631539


Top local SHAP contributors


,Feature,FeatureValue,SHAPValue,AbsSHAP
16,RankDifference,699.000000,-0.582332,0.582332
30,SurfaceELODifference,205.915889,0.431713,0.431713
29,ELODifference,161.860932,0.263157,0.263157
8,A_RankingCareerHigh,3.000000,0.194643,0.194643
65,A_BreakPointsConvertedPct52W,0.600000,-0.176021,0.176021
6,A_CurrentRanking,738.000000,-0.123650,0.123650
2,A_player_age,32.400000,-0.082821,0.082821
11,B_WeekSinceCareerHigh,21.000000,0.079031,0.079031
15,B_RankSTD52W,3.206212,-0.074108,0.074108
55,A_FirstServeInPct52W,0.573034,-0.072728,0.072728


### 12.2 Case Study 2: Clay-Court Close-Ranking Match

This case focuses on a clay-court match between two closely ranked players. Because ranking difference provides very little separation in this example, the case is used to investigate whether local SHAP explanations highlight more specific pre-match indicators, particularly return-related and surface-context variables.

In [62]:
case_norrie_meta = get_case_metadata(CASE_NORRIE)
case_norrie_stats = get_case_tennis_stats(CASE_NORRIE)
case_norrie_shap = get_local_shap_table(CASE_NORRIE, top_n=20)

print("Match metadata")
display(case_norrie_meta)

print("Selected tennis performance features")
display(case_norrie_stats)

print("Top local SHAP contributors")
display(case_norrie_shap)

Match metadata


,tourney_date,A_surface,A_player_name,B_player_name,A_CurrentRanking,B_CurrentRanking,RankDifference,A_ELO,B_ELO,ELODifference,A_SurfaceELO,B_SurfaceELO,SurfaceELODifference,pred_prob_A_win,actual_A_win
8726,2024-04-15,Clay,Cameron Norrie,Tomas Martin Etcheverry,31.0,30.0,1.0,1690.427482,1648.864146,41.563336,1694.098713,1636.484242,57.614471,0.600946,0


Selected tennis performance features


,A_player_name,B_player_name,A_surface,A_AceRateL15,B_AceRateL15,A_FirstServeWinPctL15,B_FirstServeWinPctL15,A_SecondServeWinPctL15,B_SecondServeWinPctL15,A_ServicePointsWonPctL15,...,A_ReturnPointsWonPctL15,B_ReturnPointsWonPctL15,A_BreakPointsConvertedPctL15,B_BreakPointsConvertedPctL15,A_BreakPointsEarnedRateL15,B_BreakPointsEarnedRateL15,A_ReturnPointsWonPct52W,B_ReturnPointsWonPct52W,A_ServicePointsWonPct52W,B_ServicePointsWonPct52W
8726,Cameron Norrie,Tomas Martin Etcheverry,Clay,0.063874,0.07423,0.709302,0.689133,0.526013,0.484664,0.632777,...,0.414229,0.376783,0.464617,0.42967,0.757901,0.618778,0.380663,0.368136,0.637705,0.637222


Top local SHAP contributors


,Feature,FeatureValue,SHAPValue,AbsSHAP
16,RankDifference,1.000000,-0.118714,0.118714
30,SurfaceELODifference,57.614471,0.115102,0.115102
8,A_RankingCareerHigh,8.000000,0.110429,0.110429
2,A_player_age,28.600000,0.092654,0.092654
11,B_WeekSinceCareerHigh,7.000000,0.079775,0.079775
49,A_ReturnPointsWonPctL15,0.414229,0.074851,0.074851
36,B_DFRateL15,0.012066,0.059576,0.059576
41,A_SecondServeWinPctL15,0.526013,0.049058,0.049058
10,A_WeekSinceCareerHigh,80.000000,0.047714,0.047714
24,B_SurfaceELO,1636.484242,-0.046749,0.046749


### 12.3 Case Study 3: Elite Close-Match Contest

This final case examines a match between two elite players with very similar rankings. The objective here is to investigate whether the model relies on a broader combination of factors, rather than a single dominant variable, when predicting the outcome of a closely matched high-level contest.

In [63]:
case_sinner_meta = get_case_metadata(CASE_SINNER)
case_sinner_stats = get_case_tennis_stats(CASE_SINNER)
case_sinner_shap = get_local_shap_table(CASE_SINNER, top_n=20)

print("Match metadata")
display(case_sinner_meta)

print("Selected tennis performance features")
display(case_sinner_stats)

print("Top local SHAP contributors")
display(case_sinner_shap)

Match metadata


,tourney_date,A_surface,A_player_name,B_player_name,A_CurrentRanking,B_CurrentRanking,RankDifference,A_ELO,B_ELO,ELODifference,A_SurfaceELO,B_SurfaceELO,SurfaceELODifference,pred_prob_A_win,actual_A_win
8433,2024-03-04,Hard,Jannik Sinner,Carlos Alcaraz,3.0,2.0,1.0,2142.003316,2017.449217,124.5541,2121.091761,1947.82895,173.262811,0.602213,0


Selected tennis performance features


,A_player_name,B_player_name,A_surface,A_AceRateL15,B_AceRateL15,A_FirstServeWinPctL15,B_FirstServeWinPctL15,A_SecondServeWinPctL15,B_SecondServeWinPctL15,A_ServicePointsWonPctL15,...,A_ReturnPointsWonPctL15,B_ReturnPointsWonPctL15,A_BreakPointsConvertedPctL15,B_BreakPointsConvertedPctL15,A_BreakPointsEarnedRateL15,B_BreakPointsEarnedRateL15,A_ReturnPointsWonPct52W,B_ReturnPointsWonPct52W,A_ServicePointsWonPct52W,B_ServicePointsWonPct52W
8433,Jannik Sinner,Carlos Alcaraz,Hard,0.082355,0.066836,0.79164,0.76694,0.613819,0.570441,0.722459,...,0.424954,0.408017,0.468273,0.391272,0.73749,0.721261,0.409724,0.422318,0.693533,0.680895


Top local SHAP contributors


,Feature,FeatureValue,SHAPValue,AbsSHAP
30,SurfaceELODifference,173.262811,0.397208,0.397208
29,ELODifference,124.554100,0.205209,0.205209
16,RankDifference,1.000000,-0.190674,0.190674
8,A_RankingCareerHigh,3.000000,0.187169,0.187169
3,B_player_age,20.800000,-0.184119,0.184119
64,B_ReturnPointsWonPct52W,0.422318,-0.171177,0.171177
9,B_RankingCareerHigh,1.000000,-0.161088,0.161088
2,A_player_age,22.500000,0.139478,0.139478
6,A_CurrentRanking,3.000000,0.138012,0.138012
62,B_ServicePointsWonPct52W,0.680895,-0.119392,0.119392


## 13. Save Case Study Outputs

The selected case-study tables are saved for later use in dissertation figures, results discussion, and appendix material.

In [64]:
case_raonic_meta.to_csv(OUTPUT_DIR / "case_raonic_metadata.csv", index=False)
case_raonic_stats.to_csv(OUTPUT_DIR / "case_raonic_stats.csv", index=False)
case_raonic_shap.to_csv(OUTPUT_DIR / "case_raonic_shap.csv", index=False)

case_norrie_meta.to_csv(OUTPUT_DIR / "case_norrie_metadata.csv", index=False)
case_norrie_stats.to_csv(OUTPUT_DIR / "case_norrie_stats.csv", index=False)
case_norrie_shap.to_csv(OUTPUT_DIR / "case_norrie_shap.csv", index=False)

case_sinner_meta.to_csv(OUTPUT_DIR / "case_sinner_metadata.csv", index=False)
case_sinner_stats.to_csv(OUTPUT_DIR / "case_sinner_stats.csv", index=False)
case_sinner_shap.to_csv(OUTPUT_DIR / "case_sinner_shap.csv", index=False)

print("Case study outputs saved to outputs folder")

Case study outputs saved to outputs folder


## 14. Summary of Local SHAP Findings

The three case studies illustrate that local explanations can reveal different kinds of predictive logic across tennis contexts.

- In the grass-court underdog example, the explanation highlights how surface-adjusted strength and serve-related factors can offset a large ranking disadvantage.
- In the clay-court close-ranking example, the explanation shows that recent return performance and surface context can help distinguish players when ranking difference is minimal.
- In the elite close-match example, the explanation reflects a more balanced combination of strength, serve, and return indicators.

These findings suggest that while broad player-strength variables dominate global importance, local SHAP explanations can still uncover more specific tennis-related drivers of prediction in context-dependent situations.

## Surface-Specific Filtered Feature Comparison for Logistic Regression and Random Forest

To support later dissertation figures, filtered tennis-specific feature importance is examined separately for grass and clay matches for Logistic Regression and Random Forest.

In [65]:
# Surface masks from test data
grass_mask = df_test["A_surface"] == "Grass"
clay_mask = df_test["A_surface"] == "Clay"

print("Grass matches:", grass_mask.sum())
print("Clay matches:", clay_mask.sum())

Grass matches: 1230
Clay matches: 3188


In [66]:
# Logistic Regression surface-specific importance
# coefficients are global, but we filter/display them against the same feature set
lr_surface_importance = pd.DataFrame({
    "Feature": model_features,
    "Importance": np.abs(lr_clf.coef_[0])
}).sort_values(by="Importance", ascending=False)

lr_surface_filtered = lr_surface_importance[
    ~lr_surface_importance["Feature"].isin(strength_features)
].reset_index(drop=True)

In [67]:
# Random Forest surface-specific importance via permutation on subsets
from sklearn.inspection import permutation_importance

X_test_grass = X_test_imputed_df.loc[grass_mask]
y_test_grass = y_test.loc[grass_mask]

X_test_clay = X_test_imputed_df.loc[clay_mask]
y_test_clay = y_test.loc[clay_mask]

rf_perm_grass = permutation_importance(
    rf_clf,
    X_test_grass,
    y_test_grass,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

rf_perm_clay = permutation_importance(
    rf_clf,
    X_test_clay,
    y_test_clay,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

rf_grass_importance = pd.DataFrame({
    "Feature": model_features,
    "Importance": rf_perm_grass.importances_mean
}).sort_values(by="Importance", ascending=False)

rf_clay_importance = pd.DataFrame({
    "Feature": model_features,
    "Importance": rf_perm_clay.importances_mean
}).sort_values(by="Importance", ascending=False)

rf_grass_filtered = rf_grass_importance[
    ~rf_grass_importance["Feature"].isin(strength_features)
].reset_index(drop=True)

rf_clay_filtered = rf_clay_importance[
    ~rf_clay_importance["Feature"].isin(strength_features)
].reset_index(drop=True)

/Users/baileyhorsley/miniforge3/envs/ds/lib/python3.11/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/Users/baileyhorsley/miniforge3/envs/ds/lib/python3.11/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/Users/baileyhorsley/miniforge3/envs/ds/lib/python3.11/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/Users/baileyhorsley/miniforge3/envs/ds/lib/python3.11/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/Users/baileyhorsley/miniforge3/envs/ds/lib/python3.11/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but Ran

In [68]:
# XGBoost SHAP by surface already existed, but here we save only LR/RF support tables needed for Notebook 6

lr_surface_filtered.to_csv(OUTPUT_DIR / "lr_surface_filtered.csv", index=False)
rf_grass_filtered.to_csv(OUTPUT_DIR / "rf_grass_filtered.csv", index=False)
rf_clay_filtered.to_csv(OUTPUT_DIR / "rf_clay_filtered.csv", index=False)

print("Saved:")
print((OUTPUT_DIR / "lr_surface_filtered.csv").resolve())
print((OUTPUT_DIR / "rf_grass_filtered.csv").resolve())
print((OUTPUT_DIR / "rf_clay_filtered.csv").resolve())

Saved:
/Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/outputs/lr_surface_filtered.csv
/Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/outputs/rf_grass_filtered.csv
/Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/outputs/rf_clay_filtered.csv


## Save Local SHAP Waterfall Plots

Waterfall plots are created for the three selected case studies to provide dissertation-ready local explanation visuals.

In [69]:
import matplotlib.pyplot as plt
import shap
X_test_reset = X_test_imputed_df.reset_index(drop=True)

In [70]:
def save_waterfall_plot(row_idx, filename, max_display=12):
    explanation = shap.Explanation(
        values=xgb_shap_values[row_idx],
        base_values=xgb_explainer.expected_value,
        data=X_test_reset.iloc[row_idx].values,
        feature_names=X_test_reset.columns.tolist()
    )
    
    plt.figure()
    shap.plots.waterfall(explanation, max_display=max_display, show=False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches="tight")
    plt.close()
    
    print(f"Saved: {OUTPUT_DIR / filename}")

In [71]:
save_waterfall_plot(6465, "raonic_waterfall.png", max_display=12)
save_waterfall_plot(8726, "norrie_waterfall.png", max_display=12)
save_waterfall_plot(8433, "sinner_waterfall.png", max_display=12)

Saved: /Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/outputs/raonic_waterfall.png
Saved: /Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/outputs/norrie_waterfall.png
Saved: /Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/outputs/sinner_waterfall.png
